# PhishGuard - Clean Rebuild v4

## Architecture
```
URL Input
   │
   ├─ normalize_url()  ← SINGLE function used EVERYWHERE
   │
   ├── CNN (character-level sequences)
   ├── Feature Extraction (15 structural features)
   │
   └── Hybrid XGBoost (16 features = 15 + CNN probability)
```

## Key Fix from v3
The `normalize_url()` function is called **before** everything — during training AND inference.
This guarantees the CNN sees identical patterns at both stages.

---
### Steps
1. Libraries
2. Load & verify dataset
3. Label conversion
4. `normalize_url()` — single preprocessing function
5. Train/test split
6. Tokenization + padding
7. CNN model build
8. CNN training & evaluation
9. Feature extraction
10. Hybrid model training & evaluation
11. Save all artifacts
12. Prediction pipeline — tested with diverse URLs

## STEP 1 — Libraries

In [1]:
pip install tensorflow xgboost scikit-learn beautifulsoup4 requests streamlit

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import re
import math
import pickle
import joblib
from urllib.parse import urlparse

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D, Dense, Dropout

from xgboost import XGBClassifier

print('All libraries imported successfully')
print('Versions:')
import tensorflow as tf
import xgboost as xgb
import sklearn
print(f'  TensorFlow: {tf.__version__}')
print(f'  XGBoost:    {xgb.__version__}')
print(f'  Sklearn:    {sklearn.__version__}')

All libraries imported successfully
Versions:
  TensorFlow: 2.21.0
  XGBoost:    3.2.0
  Sklearn:    1.7.2


## STEP 2 — Load & Verify Dataset

In [3]:
df = pd.read_csv('malicious_phish.csv')

print('Shape:', df.shape)
print()
print('First 5 rows:')
print(df.head())
print()
print('Class distribution:')
print(df['type'].value_counts())
print()
print('Null values:', df.isnull().sum().sum())

# Sanity check — expected ~651k rows
assert df.shape[0] > 600000, 'Dataset seems too small — check the CSV file'
assert 'url' in df.columns, 'Missing url column'
assert 'type' in df.columns, 'Missing type column'
print()
print('STEP 2 PASSED')

Shape: (651191, 2)

First 5 rows:
                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement

Class distribution:
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64

Null values: 0

STEP 2 PASSED


## STEP 3 — Label Conversion

In [4]:
df['label'] = df['type'].apply(lambda x: 0 if x == 'benign' else 1)

print('Label distribution:')
print(df['label'].value_counts())
print()

# Keep only what we need
df = df[['url', 'label']].copy()

# Verify label encoding
assert set(df['label'].unique()) == {0, 1}, 'Labels should only be 0 and 1'
assert df['label'].value_counts()[0] > 400000, 'Expected ~428k benign'
assert df['label'].value_counts()[1] > 200000, 'Expected ~223k malicious'

print('Label 0 = Legitimate:', df['label'].value_counts()[0])
print('Label 1 = Malicious:', df['label'].value_counts()[1])
print()
print('STEP 3 PASSED')

Label distribution:
label
0    428103
1    223088
Name: count, dtype: int64

Label 0 = Legitimate: 428103
Label 1 = Malicious: 223088

STEP 3 PASSED


## STEP 4 — normalize_url()

**This is the most important fix from v3.**

This single function is called:
- When preparing training data
- When predicting a new URL

Never bypass it. Never call CNN or feature extraction on a raw URL.

In [5]:
def normalize_url(url):
    """
    Normalize a URL to a consistent format.
    This function MUST be used both during training and inference.
    
    Rules:
    - Lowercase
    - Strip protocol (http://, https://)
    - Strip www.
    - Keep only valid URL characters
    """
    url = str(url).lower().strip()
    
    # Remove protocol
    url = re.sub(r'^https?://', '', url)
    
    # Remove www.
    url = re.sub(r'^www\.', '', url)
    
    # Keep only valid URL characters
    url = re.sub(r'[^a-z0-9\-._/:?=&%@#]', '', url)
    
    return url


# --- TEST normalize_url ---
test_cases = [
    ('google.com',                     'google.com'),
    ('http://google.com',              'google.com'),
    ('https://www.google.com',         'google.com'),
    ('HTTP://WWW.GOOGLE.COM/path',     'google.com/path'),
    ('https://paypal-login.com/alert', 'paypal-login.com/alert'),
]

print('Testing normalize_url():')
all_passed = True
for raw, expected in test_cases:
    result = normalize_url(raw)
    status = 'PASS' if result == expected else 'FAIL'
    if status == 'FAIL':
        all_passed = False
    print(f'  [{status}]  "{raw}"  →  "{result}"  (expected: "{expected}")')

assert all_passed, 'normalize_url() test cases failed'
print()
print('STEP 4 PASSED')

Testing normalize_url():
  [PASS]  "google.com"  →  "google.com"  (expected: "google.com")
  [PASS]  "http://google.com"  →  "google.com"  (expected: "google.com")
  [PASS]  "https://www.google.com"  →  "google.com"  (expected: "google.com")
  [PASS]  "HTTP://WWW.GOOGLE.COM/path"  →  "google.com/path"  (expected: "google.com/path")
  [PASS]  "https://paypal-login.com/alert"  →  "paypal-login.com/alert"  (expected: "paypal-login.com/alert")

STEP 4 PASSED


## STEP 5 — Apply Normalization & Train/Test Split

In [6]:
# Apply normalization to the entire dataset
df['url'] = df['url'].apply(normalize_url)

print('Sample normalized URLs:')
print(df['url'].head(10).to_string())
print()

# Verify no empty URLs
empty_count = (df['url'].str.len() == 0).sum()
print(f'Empty URLs after normalization: {empty_count}')
if empty_count > 0:
    df = df[df['url'].str.len() > 0]
    print(f'Removed empty URLs. New shape: {df.shape}')

# Train/test split — stratified to preserve class ratio
X = df['url'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training samples: {len(X_train)}')
print(f'Testing samples:  {len(X_test)}')

# Verify stratification
train_ratio = y_train.mean()
test_ratio = y_test.mean()
print(f'Malicious ratio — Train: {train_ratio:.3f}  Test: {test_ratio:.3f}')
assert abs(train_ratio - test_ratio) < 0.005, 'Stratification failed'

print()
print('STEP 5 PASSED')

Sample normalized URLs:
0                                     br-icloud.com.br
1                  mp3raid.com/music/krizz_kaliko.html
2                      bopsecrets.org/rexroth/cr/1.htm
3    garage-pirenne.be/index.php?option=com_content...
4    adventure-nicaragua.net/index.php?option=com_m...
5    buzzfil.net/m/show-art/ils-etaient-loin-de-s-i...
6        espn.go.com/nba/player/_/id/3457/brandon-rush
7       yourbittorrent.com/?q=anthony-hamilton-soulife
8                    pashminaonline.com/pure-pashminas
9        allmusic.com/album/crazy-from-the-heat-r16990

Empty URLs after normalization: 2
Removed empty URLs. New shape: (651189, 2)
Training samples: 520951
Testing samples:  130238
Malicious ratio — Train: 0.343  Test: 0.343

STEP 5 PASSED


## STEP 6 — Tokenization & Padding

In [7]:
MAX_LEN = 150  # covers >95% of URLs in dataset

# Fit tokenizer on training set ONLY
tokenizer = Tokenizer(char_level=True, lower=True)
tokenizer.fit_on_texts(X_train)

VOCAB_SIZE = len(tokenizer.word_index) + 1
print(f'Vocabulary size: {VOCAB_SIZE} characters')
print(f'Characters: {list(tokenizer.word_index.keys())}')

# Convert to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

# Check a sample
print(f'\nSample URL: "{X_train[0]}"|')
print(f'Sequence:   {X_train_seq[0]}')
print(f'Length:     {len(X_train_seq[0])}')

# Pad sequences
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f'\nPadded shapes — Train: {X_train_pad.shape}  Test: {X_test_pad.shape}')

assert X_train_pad.shape[1] == MAX_LEN, 'Padding length mismatch'
assert X_test_pad.shape[1]  == MAX_LEN, 'Padding length mismatch'

# Save tokenizer immediately
with open('char_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print('Tokenizer saved to char_tokenizer.pkl')

print()
print('STEP 6 PASSED')

Vocabulary size: 48 characters
Characters: ['e', 'o', 'a', 'i', 't', 'c', 'n', 's', 'r', '/', 'm', 'l', '.', 'd', 'p', '-', 'h', 'u', 'b', 'g', '1', 'f', '0', 'w', '2', 'y', '=', 'k', '3', 'v', '8', '%', '5', '9', '4', '_', '7', '6', '&', 'x', 'j', 'z', '?', 'q', ':', '@', '#']

Sample URL: "ca.linkedin.com/pub/francis-lessard/25/a79/3b5"|
Sequence:   [6, 3, 13, 12, 4, 7, 28, 1, 14, 4, 7, 13, 6, 2, 11, 10, 15, 18, 19, 10, 22, 9, 3, 7, 6, 4, 8, 16, 12, 1, 8, 8, 3, 9, 14, 10, 25, 33, 10, 3, 37, 34, 10, 29, 19, 33]
Length:     46

Padded shapes — Train: (520951, 150)  Test: (130238, 150)
Tokenizer saved to char_tokenizer.pkl

STEP 6 PASSED


## STEP 7 — CNN Model Architecture

In [8]:
cnn_model = Sequential([
    Embedding(VOCAB_SIZE, 64, input_length=MAX_LEN),
    Conv1D(128, 5, activation='relu'),
    MaxPooling1D(2),
    Conv1D(64, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

cnn_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

cnn_model.summary()

# Verify last layer is a sigmoid Dense(1)
last_layer = cnn_model.layers[-1]
assert isinstance(last_layer, Dense), 'Last layer must be Dense'
assert last_layer.units == 1, 'Output Dense layer must have 1 unit'
assert last_layer.activation.__name__ == 'sigmoid', 'Output activation must be sigmoid'

print()
print('STEP 7 PASSED')

C:\Users\saish\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d (Conv1D)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ ?                           │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


STEP 7 PASSED


## STEP 8 — CNN Training & Evaluation

In [9]:
history = cnn_model.fit(
    X_train_pad, y_train,
    epochs=5,
    batch_size=512,
    validation_data=(X_test_pad, y_test),
    verbose=1
)

# Evaluate
loss, acc = cnn_model.evaluate(X_test_pad, y_test, verbose=0)
print(f'\nCNN Test Accuracy: {acc:.4f}')
print(f'CNN Test Loss:     {loss:.4f}')

assert acc > 0.88, f'CNN accuracy too low: {acc}. Check data or model.'

# Save CNN model
cnn_model.save('url_cnn_model.keras')
print('CNN model saved to url_cnn_model.keras')

print()
print('STEP 8 PASSED')

Epoch 1/5
1018/1018 ━━━━━━━━━━━━━━━━━━━━ 145s 139ms/step - accuracy: 0.8743 - loss: 0.2941 - val_accuracy: 0.9145 - val_loss: 0.2042
Epoch 2/5
1018/1018 ━━━━━━━━━━━━━━━━━━━━ 148s 145ms/step - accuracy: 0.9201 - loss: 0.1967 - val_accuracy: 0.9266 - val_loss: 0.1786
Epoch 3/5
1018/1018 ━━━━━━━━━━━━━━━━━━━━ 160s 157ms/step - accuracy: 0.9308 - loss: 0.1717 - val_accuracy: 0.9306 - val_loss: 0.1692
Epoch 4/5
1018/1018 ━━━━━━━━━━━━━━━━━━━━ 148s 145ms/step - accuracy: 0.9374 - loss: 0.1569 - val_accuracy: 0.9340 - val_loss: 0.1628
Epoch 5/5
1018/1018 ━━━━━━━━━━━━━━━━━━━━ 135s 133ms/step - accuracy: 0.9408 - loss: 0.1477 - val_accuracy: 0.9348 - val_loss: 0.1603

CNN Test Accuracy: 0.9348
CNN Test Loss:     0.1603
CNN model saved to url_cnn_model.keras

STEP 8 PASSED


## STEP 9 — Feature Extraction

In [10]:
# Define a consistent feature order — never change this after training
FEATURE_NAMES = [
    'url_length', 'domain_length', 'path_length',
    'dot_count', 'hyphen_count', 'digit_count',
    'subdomain_count', 'directory_count', 'query_length',
    'special_char_count', 'has_at_symbol', 'has_ip',
    'suspicious_tld', 'suspicious_word', 'entropy'
]

SUSPICIOUS_TLDS   = {'ru', 'tk', 'ml', 'ga', 'cf', 'gq', 'xyz', 'pw'}
SUSPICIOUS_WORDS  = {'login', 'verify', 'secure', 'account', 'update', 'bank', 'alert', 'confirm'}


def extract_url_features(url):
    """
    Extract 15 structural features from a normalized URL.
    Input must already be normalized via normalize_url().
    Returns a list of floats in FEATURE_NAMES order.
    """
    # For urlparse we temporarily add http://
    parse_url = 'http://' + url if not url.startswith('http') else url
    parsed = urlparse(parse_url)
    
    domain = parsed.netloc
    path   = parsed.path
    query  = parsed.query
    
    # Entropy calculation
    if len(url) > 0:
        prob = [url.count(c) / len(url) for c in set(url)]
        entropy = -sum(p * math.log2(p) for p in prob)
    else:
        entropy = 0.0
    
    features = [
        len(url),                                                            # url_length
        len(domain),                                                         # domain_length
        len(path),                                                           # path_length
        url.count('.'),                                                      # dot_count
        url.count('-'),                                                      # hyphen_count
        sum(c.isdigit() for c in url),                                       # digit_count
        domain.count('.'),                                                   # subdomain_count
        path.count('/'),                                                     # directory_count
        len(query),                                                          # query_length
        len(re.findall(r'[!@#$%^&*(),?":{}|<>]', url)),                     # special_char_count
        1 if '@' in url else 0,                                              # has_at_symbol
        1 if re.search(r'\d+\.\d+\.\d+\.\d+', domain) else 0,             # has_ip
        1 if domain.split('.')[-1] in SUSPICIOUS_TLDS else 0,               # suspicious_tld
        1 if any(w in url for w in SUSPICIOUS_WORDS) else 0,                # suspicious_word
        entropy,                                                             # entropy
    ]
    
    return features


# --- TEST feature extraction ---
print('Testing extract_url_features():')

f = extract_url_features('paypal-login-security-alert.com')
assert len(f) == len(FEATURE_NAMES), f'Expected {len(FEATURE_NAMES)} features, got {len(f)}'
print(f'  Feature count: {len(f)} ✓')
for name, val in zip(FEATURE_NAMES, f):
    print(f'  {name:25s}: {val}')

# Verify some expected values for the test URL
feat_dict = dict(zip(FEATURE_NAMES, f))
assert feat_dict['hyphen_count'] == 3, 'Expected 3 hyphens in paypal-login-security-alert.com'
assert feat_dict['suspicious_word'] == 1, 'Expected suspicious_word=1'
assert feat_dict['suspicious_tld'] == 0, 'Expected suspicious_tld=0 for .com'
print()

f2 = extract_url_features('bank-alert.ru')
feat_dict2 = dict(zip(FEATURE_NAMES, f2))
assert feat_dict2['suspicious_tld'] == 1, 'Expected suspicious_tld=1 for .ru'
assert feat_dict2['suspicious_word'] == 1, 'Expected suspicious_word=1 for bank'
print('  .ru suspicious TLD check ✓')

print()
print('STEP 9 FUNCTION DEFINED AND TESTED')

Testing extract_url_features():
  Feature count: 15 ✓
  url_length               : 31
  domain_length            : 31
  path_length              : 0
  dot_count                : 1
  hyphen_count             : 3
  digit_count              : 0
  subdomain_count          : 1
  directory_count          : 0
  query_length             : 0
  special_char_count       : 0
  has_at_symbol            : 0
  has_ip                   : 0
  suspicious_tld           : 0
  suspicious_word          : 1
  entropy                  : 3.977916874693636

  .ru suspicious TLD check ✓

STEP 9 FUNCTION DEFINED AND TESTED


In [11]:
# Extract features for all samples
print('Extracting features from training set...')
Xf_train = np.array([extract_url_features(url) for url in X_train], dtype=np.float32)

print('Extracting features from test set...')
Xf_test  = np.array([extract_url_features(url) for url in X_test],  dtype=np.float32)

print(f'\nFeature matrix shapes — Train: {Xf_train.shape}  Test: {Xf_test.shape}')

# Verify shapes
assert Xf_train.shape[0] == len(X_train), 'Train feature row count mismatch'
assert Xf_test.shape[0]  == len(X_test),  'Test feature row count mismatch'
assert Xf_train.shape[1] == len(FEATURE_NAMES), 'Feature column count mismatch'

# Verify no NaN/inf
assert not np.any(np.isnan(Xf_train)), 'NaN found in training features'
assert not np.any(np.isinf(Xf_train)), 'Inf found in training features'

print('No NaN or Inf values ✓')
print()
print('STEP 9 PASSED')

Extracting features from training set...
Extracting features from test set...

Feature matrix shapes — Train: (520951, 15)  Test: (130238, 15)
No NaN or Inf values ✓

STEP 9 PASSED


## STEP 10 — Build Hybrid Features & Train XGBoost

In [12]:
# Get CNN probabilities for TRAINING data
print('Getting CNN probabilities for training set...')
cnn_train_probs = cnn_model.predict(X_train_pad, batch_size=512, verbose=1)
print(f'CNN train probs shape: {cnn_train_probs.shape}')

# Get CNN probabilities for TEST data
print('Getting CNN probabilities for test set...')
cnn_test_probs = cnn_model.predict(X_test_pad, batch_size=512, verbose=1)
print(f'CNN test probs shape: {cnn_test_probs.shape}')

# Build hybrid feature matrices: [15 features + 1 CNN prob] = 16 features
X_train_hybrid = np.hstack((Xf_train, cnn_train_probs))
X_test_hybrid  = np.hstack((Xf_test,  cnn_test_probs))

print(f'\nHybrid shapes — Train: {X_train_hybrid.shape}  Test: {X_test_hybrid.shape}')

assert X_train_hybrid.shape[1] == len(FEATURE_NAMES) + 1, 'Expected 16 hybrid features'
print('Hybrid feature count = 16 ✓')

Getting CNN probabilities for training set...
1018/1018 ━━━━━━━━━━━━━━━━━━━━ 52s 51ms/step
CNN train probs shape: (520951, 1)
Getting CNN probabilities for test set...
255/255 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step
CNN test probs shape: (130238, 1)

Hybrid shapes — Train: (520951, 16)  Test: (130238, 16)
Hybrid feature count = 16 ✓


In [13]:
# Train XGBoost hybrid model
print('Training hybrid XGBoost model...')

hybrid_model = XGBClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    eval_metric='logloss'
)

hybrid_model.fit(
    X_train_hybrid, y_train,
    eval_set=[(X_test_hybrid, y_test)],
    verbose=50
)

# Evaluate
hybrid_preds = hybrid_model.predict(X_test_hybrid)
acc = accuracy_score(y_test, hybrid_preds)
print(f'\nHybrid Model Accuracy: {acc:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, hybrid_preds, target_names=['Legitimate', 'Malicious']))

assert acc > 0.90, f'Hybrid accuracy too low: {acc}'

# Save hybrid model
joblib.dump(hybrid_model, 'hybrid_url_model.pkl')
print('Hybrid model saved to hybrid_url_model.pkl')

print()
print('STEP 10 PASSED')

Training hybrid XGBoost model...
[0]	validation_0-logloss:0.62531
[50]	validation_0-logloss:0.17609
[100]	validation_0-logloss:0.15175
[150]	validation_0-logloss:0.15058
[200]	validation_0-logloss:0.15021
[250]	validation_0-logloss:0.14982
[300]	validation_0-logloss:0.14943
[350]	validation_0-logloss:0.14920
[399]	validation_0-logloss:0.14899

Hybrid Model Accuracy: 0.9398

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.95      0.96      0.95     85621
   Malicious       0.93      0.90      0.91     44617

    accuracy                           0.94    130238
   macro avg       0.94      0.93      0.93    130238
weighted avg       0.94      0.94      0.94    130238

Hybrid model saved to hybrid_url_model.pkl

STEP 10 PASSED


## STEP 11 — Save All Artifacts & Config

In [14]:
# Save the config: constants that must match at inference time
config = {
    'MAX_LEN': MAX_LEN,
    'VOCAB_SIZE': VOCAB_SIZE,
    'FEATURE_NAMES': FEATURE_NAMES,
    'SUSPICIOUS_TLDS': list(SUSPICIOUS_TLDS),
    'SUSPICIOUS_WORDS': list(SUSPICIOUS_WORDS),
}

with open('phishguard_config.pkl', 'wb') as f:
    pickle.dump(config, f)

print('Saved files:')
import os
for fname in ['url_cnn_model.keras', 'char_tokenizer.pkl', 'hybrid_url_model.pkl', 'phishguard_config.pkl']:
    exists = os.path.exists(fname)
    size_kb = os.path.getsize(fname) / 1024 if exists else 0
    status = f'{size_kb:.1f} KB' if exists else 'MISSING'
    print(f'  {fname:35s}  {status}')

print()
print('STEP 11 PASSED')

Saved files:
  url_cnn_model.keras                  1088.4 KB
  char_tokenizer.pkl                   1.8 KB
  hybrid_url_model.pkl                 3614.7 KB
  phishguard_config.pkl                0.4 KB

STEP 11 PASSED


## STEP 12 — Prediction Pipeline

This is the final test. Load everything from disk and run predictions.
The pipeline here must match the training pipeline exactly.

In [15]:
# Load everything from disk — simulating fresh inference environment
from tensorflow.keras.models import load_model

loaded_cnn    = load_model('url_cnn_model.keras')
loaded_hybrid = joblib.load('hybrid_url_model.pkl')

with open('char_tokenizer.pkl', 'rb') as f:
    loaded_tokenizer = pickle.load(f)

with open('phishguard_config.pkl', 'rb') as f:
    loaded_config = pickle.load(f)

INFER_MAX_LEN       = loaded_config['MAX_LEN']
INFER_FEATURE_NAMES = loaded_config['FEATURE_NAMES']

print('All models and config loaded from disk')
print(f'MAX_LEN = {INFER_MAX_LEN}')
print(f'Features = {INFER_FEATURE_NAMES}')

All models and config loaded from disk
MAX_LEN = 150
Features = ['url_length', 'domain_length', 'path_length', 'dot_count', 'hyphen_count', 'digit_count', 'subdomain_count', 'directory_count', 'query_length', 'special_char_count', 'has_at_symbol', 'has_ip', 'suspicious_tld', 'suspicious_word', 'entropy']


In [16]:
def predict_url(url, verbose=True):
    """
    Predict whether a URL is phishing or legitimate.
    
    Pipeline:
    1. normalize_url()               ← same as training
    2. extract_url_features()        ← same function as training
    3. tokenize + pad               ← same tokenizer as training
    4. CNN probability
    5. Hybrid XGBoost prediction
    """
    # Step 1: Normalize
    norm = normalize_url(url)
    
    # Step 2: Extract features from the NORMALIZED url
    features = extract_url_features(norm)
    feature_vector = np.array(features, dtype=np.float32).reshape(1, -1)
    
    # Step 3: Tokenize + pad the NORMALIZED url
    seq = loaded_tokenizer.texts_to_sequences([norm])
    padded = pad_sequences(seq, maxlen=INFER_MAX_LEN, padding='post', truncating='post')
    
    # Step 4: CNN probability
    cnn_prob = loaded_cnn.predict(padded, verbose=0)
    
    # Step 5: Hybrid prediction
    hybrid_input = np.hstack((feature_vector, cnn_prob))
    pred = loaded_hybrid.predict(hybrid_input)[0]
    prob = loaded_hybrid.predict_proba(hybrid_input)[0]
    confidence = prob[int(pred)]
    
    label = 'PHISHING / MALICIOUS' if pred == 1 else 'LEGITIMATE'
    
    if verbose:
        print(f'URL:         {url}')
        print(f'Normalized:  {norm}')
        print(f'CNN prob:    {cnn_prob[0][0]:.4f}')
        print(f'Prediction:  {label}')
        print(f'Confidence:  {confidence:.4f}')
        print()
    
    hybrid_prob = float(prob[1])  # P(class=1, phishing)
    return {'label': label, 'pred': int(pred), 'confidence': float(confidence), 'cnn_prob': float(cnn_prob[0][0]), 'hybrid_prob': hybrid_prob}

print('predict_url() function defined')


predict_url() function defined


In [17]:
# Test with a comprehensive set of URLs
# Format: (url, expected_label)  — 0=legitimate, 1=phishing
test_urls = [
    # Clear legitimate sites
    ('google.com',                                       0),
    ('github.com',                                       0),
    ('youtube.com/watch?v=abc123',                       0),
    ('https://www.amazon.com/products',                  0),
    ('https://www.geeksforgeeks.org/python-tutorial',    0),
    
    # Clear phishing URLs
    ('paypal-login-security-alert.com',                  1),
    ('bank-verification-alert.ru',                       1),
    ('amazon-secure-login-update.tk',                    1),
    ('facebook-account-verify-login.net',                1),
    ('secure-bank-login-alert.ml',                       1),
]

print('=' * 65)
print('PREDICTION TEST RESULTS')
print('=' * 65)

results = []
for url, expected in test_urls:
    result = predict_url(url)
    correct = result['pred'] == expected
    results.append(correct)

correct_count = sum(results)
print('=' * 65)
print(f'INFERENCE TEST: {correct_count}/{len(test_urls)} correct')

if correct_count < len(test_urls) * 0.7:
    print('WARNING: Inference accuracy below 70% on test cases.')
    print('This usually means the training/inference pipeline still has a mismatch.')
else:
    print('Inference pipeline working correctly.')

print()
print('STEP 12 COMPLETE')

PREDICTION TEST RESULTS
URL:         google.com
Normalized:  google.com
CNN prob:    0.9977
Prediction:  PHISHING / MALICIOUS
Confidence:  0.9932

URL:         github.com
Normalized:  github.com
CNN prob:    0.9975
Prediction:  PHISHING / MALICIOUS
Confidence:  0.9954

URL:         youtube.com/watch?v=abc123
Normalized:  youtube.com/watch?v=abc123
CNN prob:    0.0000
Prediction:  LEGITIMATE
Confidence:  0.9994

URL:         https://www.amazon.com/products
Normalized:  amazon.com/products
CNN prob:    0.3004
Prediction:  LEGITIMATE
Confidence:  0.6900

URL:         https://www.geeksforgeeks.org/python-tutorial
Normalized:  geeksforgeeks.org/python-tutorial
CNN prob:    0.4639
Prediction:  LEGITIMATE
Confidence:  0.7145

URL:         paypal-login-security-alert.com
Normalized:  paypal-login-security-alert.com
CNN prob:    0.9996
Prediction:  PHISHING / MALICIOUS
Confidence:  0.9938

URL:         bank-verification-alert.ru
Normalized:  bank-verification-alert.ru
CNN prob:    0.9999
Predic

---
# ═══════════════════════════════════════
# MODULE 2 — HTML CONTENT ANALYSIS MODEL
# ═══════════════════════════════════════

## Architecture
```
URL
 │
 └── BeautifulSoup + requests  (web scraping)
          │
     Extract visible text
          │
     Word-level Tokenizer
          │
     Embedding → BiLSTM → Dense
          │
     Phishing probability
```

## Dataset
```
HTML Content Dataset/
    Genuine Sites/    ← .txt files, one per site, label = 0
    Phishing Sites/   ← .txt files, one per site, label = 1
```

## Steps
13. HTML libraries
14. Load HTML dataset from folders
15. Text cleaning
16. Train/test split
17. Word tokenizer + padding
18. BiLSTM model
19. Training & evaluation
20. Save HTML model artifacts
21. Web scraping function (BeautifulSoup)
22. HTML model prediction pipeline — tested
23. Combined prediction (URL model + HTML model)
24. Streamlit GUI

## STEP 13 — HTML Model Libraries

In [18]:
import os
import requests
from bs4 import BeautifulSoup

from tensorflow.keras.layers import LSTM, Bidirectional, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping

print('HTML model libraries imported')

# Verify BeautifulSoup is available
import bs4
print(f'BeautifulSoup version: {bs4.__version__}')
print()
print('STEP 13 PASSED')

HTML model libraries imported
BeautifulSoup version: 4.14.3

STEP 13 PASSED


In [19]:
# ── DATASET DIAGNOSTIC ──────────────────────────────────────────
# Run this first to see exactly what files are in your folders.
# This tells you if any extensions are missing from SUPPORTED_EXTS.

import os
from collections import Counter

DATASET_ROOT    = 'html_content'
GENUINE_FOLDER  = os.path.join(DATASET_ROOT, 'genuine_site_0')
PHISHING_FOLDER = os.path.join(DATASET_ROOT, 'phishing_site_1')

def scan_folder(folder):
    files = os.listdir(folder)
    ext_counts = Counter(os.path.splitext(f)[1].lower() for f in files)
    print(f'  Total files : {len(files)}')
    print(f'  Extensions  :')
    for ext, cnt in ext_counts.most_common():
        ext_label = ext if ext else '(no extension)'
        print(f'    {ext_label:15s}  {cnt} files')
    # Show a few actual filenames so we can see the pattern
    print(f'  Sample filenames:')
    for f in sorted(files)[:5]:
        print(f'    {f}')
    return files, ext_counts

print('GENUINE SITES FOLDER:')
g_files, g_exts = scan_folder(GENUINE_FOLDER)
print()
print('PHISHING SITES FOLDER:')
p_files, p_exts = scan_folder(PHISHING_FOLDER)
print()

# Tell the user which extensions will be loaded
SUPPORTED_EXTS = {'.txt', '.html', '.htm', '.mhtml', '.mht'}
all_exts = set(g_exts.keys()) | set(p_exts.keys())
missing  = all_exts - SUPPORTED_EXTS - {''}
if missing:
    print(f'NOTE: These extensions are present but not yet in SUPPORTED_EXTS:')
    for ext in sorted(missing):
        total = g_exts.get(ext, 0) + p_exts.get(ext, 0)
        print(f'  {ext:15s}  {total} files  ← ADD THIS to SUPPORTED_EXTS in Step 14 if you want these loaded')
else:
    print('All file extensions in your folders are supported. ✓')

expected = sum(g_exts.get(e, 0) for e in SUPPORTED_EXTS) + sum(p_exts.get(e, 0) for e in SUPPORTED_EXTS)
print(f'\nExpected samples to load: ~{expected} (minus any near-empty files)')


GENUINE SITES FOLDER:
  Total files : 1311
  Extensions  :
    .html            1194 files
    .txt             117 files
  Sample filenames:
    123people.com_141.txt
    abodedublin.com_122.txt
    absoluteastronomy.com_11.txt
    absoluteastronomy.com_111.txt
    absoluteastronomy.com_198.txt

PHISHING SITES FOLDER:
  Total files : 552
  Extensions  :
    .html            504 files
    .txt             48 files
  Sample filenames:
    abctpia-gid.com_77.txt
    astuin.org_48.txt
    austinheights.edu.my_45.txt
    babymode.com.au_160.txt
    bazurashop.com_0.txt

All file extensions in your folders are supported. ✓

Expected samples to load: ~1863 (minus any near-empty files)


## STEP 14 — Load HTML Dataset

Reads all `.txt` files from `HTML Content Dataset/Genuine Sites/` and `HTML Content Dataset/Phishing Sites/`.
Each file contains the scraped text content of one webpage.

In [20]:
DATASET_ROOT    = 'html_content'
GENUINE_FOLDER  = os.path.join(DATASET_ROOT, 'genuine_site_0')
PHISHING_FOLDER = os.path.join(DATASET_ROOT, 'phishing_site_1')

SUPPORTED_EXTENSIONS = ('.txt', '.html', '.htm')

def extract_text_from_file(fpath):
    """
    Read a .txt, .html, or .htm file and return clean visible text.
    HTML/HTM files are parsed with BeautifulSoup to strip tags first.
    Returns empty string on failure.
    """
    try:
        with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
            raw = f.read()
    except Exception as e:
        return ''

    ext = os.path.splitext(fpath)[1].lower()
    if ext in ('.html', '.htm'):
        soup = BeautifulSoup(raw, 'html.parser')
        for tag in soup(['script', 'style', 'head', 'meta', 'noscript']):
            tag.decompose()
        text = soup.get_text(separator=' ')
    else:
        text = raw  # plain .txt — use as-is

    return text.strip()


def load_folder(folder, label):
    """Load all supported files from a folder. Returns list of (text, label) tuples."""
    samples = []
    all_files = os.listdir(folder)
    supported = [f for f in all_files if f.lower().endswith(SUPPORTED_EXTENSIONS)]
    skipped = 0

    for fname in supported:
        fpath = os.path.join(folder, fname)
        text = extract_text_from_file(fpath)
        if len(text) > 20:
            samples.append((text, label))
        else:
            skipped += 1

    print(f'  {folder}')
    print(f'    Total files found : {len(all_files)}')
    print(f'    Supported (.txt/.html/.htm) : {len(supported)}')
    print(f'    Loaded (non-empty): {len(samples)}')
    print(f'    Skipped (empty)   : {skipped}')
    return samples


# Verify folders exist
assert os.path.isdir(GENUINE_FOLDER),  f'Folder not found: {GENUINE_FOLDER}'
assert os.path.isdir(PHISHING_FOLDER), f'Folder not found: {PHISHING_FOLDER}'

print('Loading Genuine Sites...')
genuine_samples  = load_folder(GENUINE_FOLDER,  label=0)
print()
print('Loading Phishing Sites...')
phishing_samples = load_folder(PHISHING_FOLDER, label=1)

print()
print(f'Genuine  samples: {len(genuine_samples)}')
print(f'Phishing samples: {len(phishing_samples)}')
print(f'Total            : {len(genuine_samples) + len(phishing_samples)}')

assert len(genuine_samples)  > 50, 'Too few genuine samples — check folder path'
assert len(phishing_samples) > 50, 'Too few phishing samples — check folder path'

# Class balance check
ratio = len(phishing_samples) / max(len(genuine_samples), 1)
print(f'Class ratio (phishing/genuine): {ratio:.2f}')
if ratio < 0.3 or ratio > 3.0:
    print('WARNING: Dataset is heavily imbalanced. Consider class_weight in training.')

# Combine
all_samples = genuine_samples + phishing_samples
html_texts  = [s[0] for s in all_samples]
html_labels = [s[1] for s in all_samples]

# Preview
print()
print('Genuine sample preview (first 300 chars):')
print(genuine_samples[0][0][:300])
print()
print('Phishing sample preview (first 300 chars):')
print(phishing_samples[0][0][:300])
print()
print('STEP 14 PASSED')


Loading Genuine Sites...
  html_content\genuine_site_0
    Total files found : 1311
    Supported (.txt/.html/.htm) : 1311
    Loaded (non-empty): 1205
    Skipped (empty)   : 106

Loading Phishing Sites...
  html_content\phishing_site_1
    Total files found : 552
    Supported (.txt/.html/.htm) : 552
    Loaded (non-empty): 490
    Skipped (empty)   : 62

Genuine  samples: 1205
Phishing samples: 490
Total            : 1695
Class ratio (phishing/genuine): 0.41

Genuine sample preview (first 300 chars):
<html lang="de" prefix="og: https://ogp.me/ns#"><head>
	<meta charset="UTF-8">
	<meta name="viewport" content="width=device-width, initial-scale=1">
	<link rel="profile" href="https://gmpg.org/xfn/11">
		<style>img:is([sizes="auto" i], [sizes^="auto," i]) { contain-intrinsic-size: 3000px 1500px }</s

Phishing sample preview (first 300 chars):
<html xml:lang="ru-ru" lang="ru-ru" dir="ltr"><head>
		<meta http-equiv="content-type" content="text/html; charset=utf-8">
		<title>404-Ошибка: 40

## STEP 15 — HTML Text Cleaning

In [21]:
def clean_html_text(text):
    """
    Clean raw webpage text for the BiLSTM model.
    Works on both pre-extracted plain text AND raw HTML markup.
    - Strip remaining HTML tags
    - Remove URLs, javascript, CSS fragments
    - Keep only alphabetic words
    - Collapse whitespace
    - Truncate to 1000 words (longer cap for HTML-rich files)
    """
    text = str(text).lower()
    text = re.sub(r'<[^>]+>',          ' ', text)   # remove HTML tags
    text = re.sub(r'https?://\S+',     ' ', text)   # remove URLs
    text = re.sub(r'\{[^}]*\}',         ' ', text)   # remove CSS blocks
    text = re.sub(r'[^a-z\s]',         ' ', text)   # keep only letters
    text = re.sub(r'\b\w{1,2}\b',      ' ', text)   # remove 1-2 char noise words
    text = re.sub(r'\s+',             ' ', text).strip()
    words = text.split()[:1000]                      # cap at 1000 words
    return ' '.join(words)


# Apply cleaning
html_texts_clean = [clean_html_text(t) for t in html_texts]

# Remove any that became empty after cleaning
before = len(html_texts_clean)
valid  = [(t, l) for t, l in zip(html_texts_clean, html_labels) if len(t.strip()) > 10]
html_texts_clean = [v[0] for v in valid]
html_labels      = [v[1] for v in valid]
after = len(html_texts_clean)

if before != after:
    print(f'Removed {before - after} empty/near-empty samples after cleaning')

lengths = [len(t.split()) for t in html_texts_clean]
print(f'Samples after cleaning  : {len(html_texts_clean)}')
print(f'Word count — min: {min(lengths)}  max: {max(lengths)}  mean: {int(sum(lengths)/len(lengths))}')
print()
print('Cleaned text preview (genuine):')
print(html_texts_clean[0][:400])
print()
print('STEP 15 PASSED')

Removed 18 empty/near-empty samples after cleaning
Samples after cleaning  : 1677
Word count — min: 2  max: 1000  mean: 371

Cleaned text preview (genuine):
img sizes auto sizes auto page not found people com this file auto generated function sessionstorage setitem json stringify catch function function return function textbaseline top font arial return foreach function function undefined typeof promise wpemojisettingssupports flag emoji supports new promise function new promise function root root where body site blocks alignleft site blocks alignrigh

STEP 15 PASSED


## STEP 16 — Train/Test Split (HTML)

In [22]:
import numpy as np
html_labels_arr = np.array(html_labels)

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    html_texts_clean,
    html_labels_arr,
    test_size=0.2,
    random_state=42,
    stratify=html_labels_arr
)

print(f'HTML Train samples: {len(Xh_train)}')
print(f'HTML Test  samples: {len(Xh_test)}')
print(f'Malicious ratio — Train: {yh_train.mean():.3f}  Test: {yh_test.mean():.3f}')

assert abs(yh_train.mean() - yh_test.mean()) < 0.05, 'Stratification failed'
print()
print('STEP 16 PASSED')

HTML Train samples: 1341
HTML Test  samples: 336
Malicious ratio — Train: 0.281  Test: 0.283

STEP 16 PASSED


## STEP 17 — Word Tokenizer & Padding (HTML)

In [23]:
HTML_MAX_WORDS  = 20000   # increased from 10k to handle larger, more varied dataset
HTML_MAX_LEN    = 400     # increased from 300 — HTML docs tend to be richer than scraped .txt

html_tokenizer = Tokenizer(
    num_words=HTML_MAX_WORDS,
    oov_token='<OOV>'
)
html_tokenizer.fit_on_texts(Xh_train)

actual_vocab = len(html_tokenizer.word_index)
print(f'Unique words in training set : {actual_vocab}')
print(f'Vocabulary cap               : {HTML_MAX_WORDS}')
coverage = min(HTML_MAX_WORDS, actual_vocab) / actual_vocab * 100
print(f'Vocabulary coverage          : {coverage:.1f}%')

# Convert to sequences
Xh_train_seq = html_tokenizer.texts_to_sequences(Xh_train)
Xh_test_seq  = html_tokenizer.texts_to_sequences(Xh_test)

# Pad
Xh_train_pad = pad_sequences(Xh_train_seq, maxlen=HTML_MAX_LEN, padding='post', truncating='post')
Xh_test_pad  = pad_sequences(Xh_test_seq,  maxlen=HTML_MAX_LEN, padding='post', truncating='post')

print(f'\nPadded shapes — Train: {Xh_train_pad.shape}  Test: {Xh_test_pad.shape}')
assert Xh_train_pad.shape[1] == HTML_MAX_LEN

# Save
with open('html_tokenizer.pkl', 'wb') as f:
    pickle.dump(html_tokenizer, f)
print('HTML tokenizer saved to html_tokenizer.pkl')
print()
print('STEP 17 PASSED')


Unique words in training set : 43922
Vocabulary cap               : 20000
Vocabulary coverage          : 45.5%

Padded shapes — Train: (1341, 400)  Test: (336, 400)
HTML tokenizer saved to html_tokenizer.pkl

STEP 17 PASSED


## STEP 18 — BiLSTM Model

BiLSTM reads the word sequence in both directions (forward + backward),
so it can detect phishing patterns regardless of where they appear in the page.

In [24]:
from tensorflow.keras.layers import LSTM, Bidirectional, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

html_model = Sequential([
    Embedding(HTML_MAX_WORDS, 128, input_length=HTML_MAX_LEN),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(128, dropout=0.3, recurrent_dropout=0.2, return_sequences=True)),
    Bidirectional(LSTM(64,  dropout=0.2, recurrent_dropout=0.2, return_sequences=False)),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(64,  activation='relu'),
    Dropout(0.3),
    Dense(1,   activation='sigmoid')
])

html_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

html_model.summary()

# Verify
assert isinstance(html_model.layers[2], Bidirectional), 'Layer 2 must be Bidirectional'
assert html_model.layers[-1].units == 1, 'Output must be single unit'
print()
print('STEP 18 PASSED')


C:\Users\saish\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ spatial_dropout1d (SpatialDropout1D) │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_1 (Bidirectional)      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


STEP 18 PASSED


## STEP 19 — BiLSTM Training & Evaluation

In [25]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights to handle imbalance
classes = np.unique(yh_train)
weights = compute_class_weight('balanced', classes=classes, y=yh_train)
class_weight_dict = dict(zip(classes.tolist(), weights.tolist()))
print(f'Class weights: {class_weight_dict}')

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

html_history = html_model.fit(
    Xh_train_pad, yh_train,
    epochs=15,
    batch_size=32,
    validation_data=(Xh_test_pad, yh_test),
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Evaluate
loss, acc = html_model.evaluate(Xh_test_pad, yh_test, verbose=0)
print(f'\nBiLSTM HTML Model Accuracy : {acc:.4f}')
print(f'BiLSTM HTML Model Loss     : {loss:.4f}')
print()
print('Classification Report:')
html_preds_raw = html_model.predict(Xh_test_pad, verbose=0)
html_preds     = (html_preds_raw > 0.5).astype(int).flatten()
print(classification_report(yh_test, html_preds, target_names=['Legitimate', 'Phishing']))

# Warn but don't hard-fail — dataset size and quality vary
if acc < 0.65:
    print(f'WARNING: Accuracy {acc:.4f} is below 65%.')
    print('Possible causes: very small dataset, highly imbalanced classes,')
    print('or HTML files with very little readable text content.')
    print('The model is still saved and usable.')
else:
    print(f'Accuracy {acc:.4f} — acceptable.')

print()
print('STEP 19 PASSED')


Class weights: {0: 0.6955394190871369, 1: 1.7785145888594165}
Epoch 1/15
42/42 ━━━━━━━━━━━━━━━━━━━━ 145s 3s/step - accuracy: 0.6406 - loss: 0.6413 - val_accuracy: 0.5714 - val_loss: 0.7143 - learning_rate: 0.0010
Epoch 2/15
42/42 ━━━━━━━━━━━━━━━━━━━━ 130s 3s/step - accuracy: 0.7770 - loss: 0.4477 - val_accuracy: 0.7560 - val_loss: 0.5710 - learning_rate: 0.0010
Epoch 3/15
42/42 ━━━━━━━━━━━━━━━━━━━━ 134s 3s/step - accuracy: 0.8688 - loss: 0.3171 - val_accuracy: 0.7500 - val_loss: 0.5180 - learning_rate: 0.0010
Epoch 4/15
42/42 ━━━━━━━━━━━━━━━━━━━━ 134s 3s/step - accuracy: 0.8911 - loss: 0.2618 - val_accuracy: 0.8036 - val_loss: 0.4847 - learning_rate: 0.0010
Epoch 5/15
42/42 ━━━━━━━━━━━━━━━━━━━━ 126s 3s/step - accuracy: 0.9157 - loss: 0.1895 - val_accuracy: 0.8244 - val_loss: 0.5930 - learning_rate: 0.0010
Epoch 6/15
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9336 - loss: 0.1476
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
42/42 ━━━━━━━━━━━━━━━━━━

## STEP 20 — Save HTML Model Artifacts

In [26]:
html_model.save('html_bilstm_model.keras')
print('BiLSTM model saved to html_bilstm_model.keras')

html_config = {
    'HTML_MAX_WORDS': HTML_MAX_WORDS,
    'HTML_MAX_LEN':   HTML_MAX_LEN,
}
with open('html_config.pkl', 'wb') as f:
    pickle.dump(html_config, f)
print('HTML config saved to html_config.pkl')

print()
print('Saved files:')
for fname in ['html_bilstm_model.keras', 'html_tokenizer.pkl', 'html_config.pkl']:
    if os.path.exists(fname):
        size_kb = os.path.getsize(fname) / 1024
        print(f'  {fname:35s}  {size_kb:.1f} KB')
    else:
        print(f'  {fname:35s}  (not found)')

print()
print('STEP 20 PASSED')


BiLSTM model saved to html_bilstm_model.keras
HTML config saved to html_config.pkl

Saved files:
  html_bilstm_model.keras              35373.0 KB
  html_tokenizer.pkl                   1733.3 KB
  html_config.pkl                      0.1 KB

STEP 20 PASSED


## STEP 21 — Web Scraping Function (BeautifulSoup)

This function is used at inference time to scrape live websites.
It is **not** used for training — training uses the pre-scraped `.txt` files.

Strategy:
- Use `requests` with a realistic browser User-Agent to avoid blocks
- Use `BeautifulSoup` to extract only visible text (no scripts, styles)
- Timeout after 10 seconds to avoid hanging
- Return empty string on failure (model will fall back to URL-only analysis)

In [27]:
import requests
from bs4 import BeautifulSoup
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

SCRAPE_TIMEOUT = 12

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 Firefox/124.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
]

def scrape_url(url):
    """
    Robust scraper. Returns (text, status, fail_reason).
    fail_reason is one of: None | 'ssl_error' | 'timeout' | 'bot_blocked'
                           | 'connection_error' | 'empty_content' | 'js_only'
    This fail_reason is used by the combiner to add a suspicion signal
    when scraping fails — because scrape failures are themselves a red flag.
    """
    url = url.strip()

    # Build candidate URLs: try https first (more common), then http
    if not url.startswith(('http://', 'https://')):
        candidates = ['https://' + url, 'http://' + url]
    elif url.startswith('http://'):
        candidates = ['https://' + url[7:], url]
    else:
        candidates = [url, 'http://' + url[8:]]

    last_error   = 'unknown'
    last_reason  = 'unknown'

    for fetch_url in candidates:
        for ua in USER_AGENTS:
            headers = {
                'User-Agent': ua,
                'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
                'Accept-Language': 'en-US,en;q=0.9',
                'Accept-Encoding': 'gzip, deflate, br',
                'Connection': 'keep-alive',
                'Upgrade-Insecure-Requests': '1',
            }

            # Try with SSL, then without SSL on same URL before moving on
            for verify_ssl in [True, False]:
                try:
                    session = requests.Session()
                    session.max_redirects = 6
                    response = session.get(
                        fetch_url,
                        headers=headers,
                        timeout=SCRAPE_TIMEOUT,
                        allow_redirects=True,
                        verify=verify_ssl,
                    )

                    if response.status_code in (403, 429, 503):
                        last_error  = f'HTTP {response.status_code}'
                        last_reason = 'bot_blocked'
                        # Don't break — try next UA, it might work
                        break

                    if response.status_code == 404:
                        last_error  = 'HTTP 404 Not Found'
                        last_reason = 'connection_error'
                        break

                    response.raise_for_status()

                    # Smart encoding detection
                    enc = (response.encoding or '').lower()
                    if enc in ('iso-8859-1', 'latin-1', 'windows-1252', ''):
                        try:
                            html = response.content.decode('utf-8')
                        except UnicodeDecodeError:
                            html = response.content.decode('latin-1', errors='replace')
                    else:
                        html = response.text

                    if not html or len(html.strip()) < 100:
                        last_error  = 'Empty/tiny response body'
                        last_reason = 'empty_content'
                        continue

                    soup = BeautifulSoup(html, 'html.parser')

                    # Remove non-content tags
                    for tag in soup(['script', 'style', 'head', 'meta', 'noscript',
                                     'header', 'footer', 'nav', 'aside', 'iframe',
                                     'svg', 'img', 'button', 'input', 'form']):
                        tag.decompose()

                    text = re.sub(r'\s+', ' ', soup.get_text(separator=' ')).strip()

                    if len(text) < 100:
                        last_error  = f'Only {len(text)} chars after parsing (JS-rendered?)'
                        last_reason = 'js_only'
                        if not verify_ssl:
                            continue   # try http fallback
                        break         # try next UA

                    ssl_note = '' if verify_ssl else ' [SSL skipped]'
                    return text, f'OK — {len(text):,} chars scraped{ssl_note}', None

                except requests.exceptions.SSLError:
                    last_error  = 'SSL certificate error'
                    last_reason = 'ssl_error'
                    if verify_ssl:
                        continue   # retry without SSL
                    break

                except requests.exceptions.Timeout:
                    last_error  = f'Timed out after {SCRAPE_TIMEOUT}s'
                    last_reason = 'timeout'
                    break   # no point retrying same URL with same timeout

                except requests.exceptions.TooManyRedirects:
                    last_error  = 'Too many redirects'
                    last_reason = 'connection_error'
                    break

                except requests.exceptions.ConnectionError as e:
                    msg = str(e)
                    if 'getaddrinfo' in msg or 'Name or service' in msg:
                        last_error  = 'Domain does not resolve (site may be down or fake)'
                        last_reason = 'connection_error'
                    else:
                        last_error  = f'Connection error: {msg[:60]}'
                        last_reason = 'connection_error'
                    break

                except requests.exceptions.HTTPError as e:
                    last_error  = f'HTTP error: {e}'
                    last_reason = 'connection_error'
                    break

                except Exception as e:
                    last_error  = f'Unexpected: {str(e)[:60]}'
                    last_reason = 'unknown'
                    break

    return '', f'SCRAPE FAILED — {last_error}', last_reason


# ── Test ──────────────────────────────────────────────────────────────────────
print('Testing scraper...')
for test_url in ['http://example.com', 'https://httpbin.org/html']:
    text, status, reason = scrape_url(test_url)
    print(f'  {test_url}')
    print(f'  Status: {status}')
    print(f'  Fail reason: {reason}')
    if text: print(f'  Preview: {text[:100]}')
    print()

print('STEP 21 PASSED')


Testing scraper...
  http://example.com
  Status: OK — 127 chars scraped [SSL skipped]
  Fail reason: None
  Preview: Example Domain This domain is for use in documentation examples without needing permission. Avoid us

  https://httpbin.org/html
  Status: OK — 3,594 chars scraped
  Fail reason: None
  Preview: Herman Melville - Moby-Dick Availing himself of the mild, summer-cool weather that now reigned in th

STEP 21 PASSED


## STEP 22 — HTML Model Prediction Function

In [28]:
# Load HTML model artifacts from disk
from tensorflow.keras.models import load_model as keras_load_model

loaded_html_model = keras_load_model('html_bilstm_model.keras')

with open('html_tokenizer.pkl', 'rb') as f:
    loaded_html_tokenizer = pickle.load(f)

with open('html_config.pkl', 'rb') as f:
    loaded_html_config = pickle.load(f)

INFER_HTML_MAX_LEN = loaded_html_config['HTML_MAX_LEN']

print('HTML model artifacts loaded from disk')

HTML model artifacts loaded from disk


In [29]:
def predict_html(url, verbose=True):
    """
    Scrape + predict. Returns prob=None on failure, but always returns fail_reason
    so the combiner can adjust the final score accordingly.
    """
    raw_text, status, fail_reason = scrape_url(url)

    if not raw_text:
        if verbose:
            print(f'Scrape FAILED: {status}')
            print(f'Fail reason:   {fail_reason}')
        return {
            'prob': None, 'pred': None, 'label': 'N/A',
            'status': status, 'scraped_chars': 0,
            'text_quality': 'scrape_failed', 'fail_reason': fail_reason
        }

    word_count = len(raw_text.split())
    if   word_count >= 500: text_quality = 'high'
    elif word_count >= 150: text_quality = 'medium'
    elif word_count >= 50:  text_quality = 'low'
    else:                   text_quality = 'very_low'

    clean_text = clean_html_text(raw_text)

    if not clean_text.strip():
        if verbose:
            print('Text empty after cleaning (likely JS-only page)')
        return {
            'prob': None, 'pred': None, 'label': 'N/A',
            'status': 'Empty after cleaning', 'scraped_chars': len(raw_text),
            'text_quality': 'empty_after_clean', 'fail_reason': 'js_only'
        }

    seq    = loaded_html_tokenizer.texts_to_sequences([clean_text])
    padded = pad_sequences(seq, maxlen=INFER_HTML_MAX_LEN, padding='post', truncating='post')
    prob   = float(loaded_html_model.predict(padded, verbose=0)[0][0])
    pred   = 1 if prob > 0.5 else 0
    label  = 'PHISHING' if pred == 1 else 'LEGITIMATE'

    if verbose:
        print(f'Scrape : {status}')
        print(f'Quality: {text_quality} ({word_count:,} words)')
        print(f'HTML prob: {prob:.4f}  →  {label}')
        print()

    return {
        'prob': prob, 'pred': pred, 'label': label,
        'status': status, 'scraped_chars': len(raw_text),
        'text_quality': text_quality, 'fail_reason': None
    }

print('predict_html() defined')
print()
print('STEP 22 PASSED')


predict_html() defined

STEP 22 PASSED


## STEP 23 — Combined Prediction

Combines URL model and HTML model into one final score.

**Weighting logic:**
- If HTML scrape succeeds: `final = 0.4 × URL_prob + 0.6 × HTML_prob`  
  (HTML content is a stronger signal when available)
- If HTML scrape fails:   `final = URL_prob`  
  (fall back to URL-only analysis)

**Verdict thresholds:**
- `>= 0.70` → PHISHING
- `0.40 – 0.69` → SUSPICIOUS
- `< 0.40` → LEGITIMATE

In [30]:
# Suspicion penalty applied when scraping fails.
# Values are ADDED to the URL model probability before thresholding.
# Rationale: sites that block scraping or have no content are more suspicious.
SCRAPE_FAIL_PENALTY = {
    'ssl_error':        0.15,   # Bad SSL = red flag (phishing sites cut corners)
    'bot_blocked':      0.10,   # Actively blocking = somewhat suspicious
    'timeout':          0.08,   # Slow/unresponsive — mild suspicion
    'js_only':          0.05,   # JS-only is common for legit SPAs too — small penalty
    'empty_content':    0.10,   # Empty page = suspicious
    'connection_error': 0.12,   # DNS failure / unreachable = suspicious
    None:               0.0,    # Unknown — no penalty
    'unknown':          0.0,
}

def combined_predict(url, verbose=True):
    """
    Full PhishGuard pipeline combining URL model + HTML model.

    COMBINING STRATEGY
    ==================
    When HTML scrape SUCCEEDS:
        final = url_prob * url_weight + html_prob * html_weight
        Weights depend on content quality:
          high/medium  ->  URL 35% + HTML 65%  (HTML is more informative)
          low          ->  URL 55% + HTML 45%  (HTML is noisy, lean on URL)
          very_low     ->  URL 65% + HTML 35%  (very little content — mostly trust URL)

    When HTML scrape FAILS:
        final = min(1.0, url_prob + SCRAPE_FAIL_PENALTY[fail_reason])
        Why: sites blocking scraping ARE more suspicious, especially for SSL errors,
        DNS failures, or actively blocking bot requests. The penalty reflects this
        without blindly calling everything phishing.
        The URL model score still anchors the result — a clearly clean URL
        like google.com gets url_prob ~0.05, so even +0.15 = 0.20 = LEGITIMATE.
        Only borderline or suspicious URLs cross the verdict thresholds.

    VERDICT THRESHOLDS
    ==================
        >= 0.75  ->  PHISHING   (HIGH risk)
        >= 0.50  ->  SUSPICIOUS (MEDIUM risk)
        >= 0.30  ->  SUSPICIOUS (LOW risk)
         < 0.30  ->  LEGITIMATE (MINIMAL risk)
    """
    if verbose:
        print('=' * 60)
        print(f'  Analysing: {url}')
        print('=' * 60)

    # --- URL Model ---
    url_result = predict_url(url, verbose=verbose)
    url_prob   = url_result['hybrid_prob']

    # --- HTML Model ---
    html_result  = predict_html(url, verbose=verbose)
    html_prob    = html_result['prob']
    text_quality = html_result.get('text_quality', 'unknown')
    fail_reason  = html_result.get('fail_reason', None)

    # --- Combine ---
    if html_prob is not None:
        # Scrape succeeded — weighted combination
        quality_weights = {
            'high':     (0.35, 0.65),
            'medium':   (0.35, 0.65),
            'low':      (0.55, 0.45),
            'very_low': (0.65, 0.35),
        }
        url_w, html_w  = quality_weights.get(text_quality, (0.50, 0.50))
        final_prob     = url_w * url_prob + html_w * html_prob
        penalty        = 0.0
        analysis_mode  = f'URL({url_w:.0%}) + HTML({html_w:.0%}) — {text_quality} quality'
    else:
        # Scrape failed — URL only + suspicion penalty
        penalty       = SCRAPE_FAIL_PENALTY.get(fail_reason, 0.0)
        final_prob    = min(1.0, url_prob + penalty)
        url_w = html_w = None
        analysis_mode = f'URL only — scrape failed ({fail_reason})'
        if penalty > 0:
            analysis_mode += f' +{penalty:.2f} penalty applied'

    # --- Verdict ---
    if   final_prob >= 0.75: verdict, risk = 'PHISHING',   'HIGH'
    elif final_prob >= 0.50: verdict, risk = 'SUSPICIOUS',  'MEDIUM'
    elif final_prob >= 0.30: verdict, risk = 'SUSPICIOUS',  'LOW'
    else:                    verdict, risk = 'LEGITIMATE',  'MINIMAL'

    result = {
        'url':           url,
        'url_prob':      url_prob,
        'html_prob':     html_prob,
        'scrape_penalty': penalty if html_prob is None else 0.0,
        'final_prob':    final_prob,
        'verdict':       verdict,
        'risk':          risk,
        'analysis_mode': analysis_mode,
        'text_quality':  text_quality,
        'fail_reason':   fail_reason,
    }

    if verbose:
        print('-' * 60)
        print(f'  VERDICT    : {verdict}  (risk: {risk})')
        print(f'  Final prob : {final_prob:.1%}')
        print(f'  URL prob   : {url_prob:.4f}')
        if html_prob is not None:
            print(f'  HTML prob  : {html_prob:.4f}')
            print(f'  Weights    : URL {url_w:.0%}  HTML {html_w:.0%}')
        else:
            print(f'  Penalty    : +{penalty:.2f}  (reason: {fail_reason})')
        print(f'  Mode       : {analysis_mode}')
        print('=' * 60)

    return result

print('combined_predict() defined with scrape-failure penalty logic')
print()
print('STEP 23 PASSED')


combined_predict() defined with scrape-failure penalty logic

STEP 23 PASSED


## STEP 24 — Streamlit GUI

Writes `phishguard_app.py` to disk — run it separately from terminal:
```
streamlit run phishguard_app.py
```

The GUI will open in your browser automatically.

In [31]:
gui_code = 'import streamlit as st\nimport numpy as np\nimport re, math, pickle, joblib, os, requests\nimport urllib3\nurllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)\nfrom urllib.parse import urlparse\nfrom bs4 import BeautifulSoup\nfrom tensorflow.keras.models import load_model\nfrom tensorflow.keras.preprocessing.sequence import pad_sequences\n\nst.set_page_config(page_title=\'PhishGuard\', page_icon=\'shield\', layout=\'centered\')\n\n@st.cache_resource\ndef load_all_models():\n    cnn   = load_model(\'url_cnn_model.keras\')\n    html  = load_model(\'html_bilstm_model.keras\')\n    xgb   = joblib.load(\'hybrid_url_model.pkl\')\n    with open(\'char_tokenizer.pkl\',\'rb\') as f: ctok = pickle.load(f)\n    with open(\'html_tokenizer.pkl\',\'rb\') as f: htok = pickle.load(f)\n    with open(\'phishguard_config.pkl\',\'rb\') as f: cfg = pickle.load(f)\n    with open(\'html_config.pkl\',\'rb\') as f: hcfg = pickle.load(f)\n    return cnn, html, xgb, ctok, htok, cfg, hcfg\n\ncnn_model, html_model, hybrid_model, char_tok, html_tok, cfg, html_cfg = load_all_models()\nMAX_LEN         = cfg[\'MAX_LEN\']\nSUSPICIOUS_TLDS = set(cfg[\'SUSPICIOUS_TLDS\'])\nSUSPICIOUS_WORDS= set(cfg[\'SUSPICIOUS_WORDS\'])\nHTML_MAX_LEN    = html_cfg[\'HTML_MAX_LEN\']\n\nSCRAPE_FAIL_PENALTY = {\n    \'ssl_error\': 0.15, \'bot_blocked\': 0.10, \'timeout\': 0.08,\n    \'js_only\': 0.05, \'empty_content\': 0.10, \'connection_error\': 0.12,\n    None: 0.0, \'unknown\': 0.0,\n}\n\nUSER_AGENTS = [\n    \'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/122.0.0.0 Safari/537.36\',\n    \'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 Firefox/124.0\',\n    \'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/122.0.0.0 Safari/537.36\',\n]\n\ndef normalize_url(url):\n    url = str(url).lower().strip()\n    url = re.sub(r\'^https?://\', \'\', url)\n    url = re.sub(r\'^www\\.\', \'\', url)\n    url = re.sub(r\'[^a-z0-9\\-._/:?=&%@#]\', \'\', url)\n    return url\n\ndef extract_url_features(url):\n    pu = \'http://\' + url if not url.startswith(\'http\') else url\n    p  = urlparse(pu)\n    d, path, q = p.netloc, p.path, p.query\n    entropy = 0.0\n    if url:\n        prb = [url.count(c)/len(url) for c in set(url)]\n        entropy = -sum(x*math.log2(x) for x in prb)\n    return [\n        len(url), len(d), len(path),\n        url.count(\'.\'), url.count(\'-\'), sum(c.isdigit() for c in url),\n        d.count(\'.\'), path.count(\'/\'), len(q),\n        len(re.findall(r\'[!@#$%^&*(),?{}|<>]\', url)),\n        1 if \'@\' in url else 0,\n        1 if re.search(r\'\\d+\\.\\d+\\.\\d+\\.\\d+\', d) else 0,\n        1 if d.split(\'.\')[-1] in SUSPICIOUS_TLDS else 0,\n        1 if any(w in url for w in SUSPICIOUS_WORDS) else 0,\n        entropy,\n    ]\n\ndef clean_html_text(text):\n    text = str(text).lower()\n    text = re.sub(r\'<[^>]+>\', \' \', text)\n    text = re.sub(r\'https?://\\S+\', \' \', text)\n    text = re.sub(r\'[^a-z\\s]\', \' \', text)\n    return \' \'.join(re.sub(r\'\\s+\', \' \', text).strip().split()[:500])\n\ndef scrape_url(url):\n    url = url.strip()\n    if not url.startswith((\'http://\',\'https://\')):\n        candidates = [\'https://\'+url, \'http://\'+url]\n    elif url.startswith(\'http://\'):\n        candidates = [\'https://\'+url[7:], url]\n    else:\n        candidates = [url, \'http://\'+url[8:]]\n    last_error, last_reason = \'unknown\', \'unknown\'\n    for fetch_url in candidates:\n        for ua in USER_AGENTS:\n            headers = {\n                \'User-Agent\': ua,\n                \'Accept\': \'text/html,application/xhtml+xml,*/*;q=0.8\',\n                \'Accept-Language\': \'en-US,en;q=0.9\',\n                \'Connection\': \'keep-alive\',\n                \'Upgrade-Insecure-Requests\': \'1\',\n            }\n            for verify_ssl in [True, False]:\n                try:\n                    s = requests.Session()\n                    s.max_redirects = 6\n                    r = s.get(fetch_url, headers=headers, timeout=12,\n                              allow_redirects=True, verify=verify_ssl)\n                    if r.status_code in (403,429,503):\n                        last_error, last_reason = f\'HTTP {r.status_code}\', \'bot_blocked\'\n                        break\n                    if r.status_code == 404:\n                        last_error, last_reason = \'HTTP 404\', \'connection_error\'\n                        break\n                    r.raise_for_status()\n                    enc = (r.encoding or \'\').lower()\n                    if enc in (\'iso-8859-1\',\'latin-1\',\'windows-1252\',\'\'):\n                        try: html = r.content.decode(\'utf-8\')\n                        except: html = r.content.decode(\'latin-1\', errors=\'replace\')\n                    else:\n                        html = r.text\n                    if not html or len(html.strip()) < 100:\n                        last_error, last_reason = \'empty body\', \'empty_content\'\n                        continue\n                    soup = BeautifulSoup(html, \'html.parser\')\n                    for tag in soup([\'script\',\'style\',\'head\',\'meta\',\'noscript\',\n                                     \'header\',\'footer\',\'nav\',\'aside\',\'iframe\',\'svg\']):\n                        tag.decompose()\n                    text = re.sub(r\'\\s+\', \' \', soup.get_text(separator=\' \')).strip()\n                    if len(text) < 100:\n                        last_error, last_reason = f\'{len(text)} chars (JS page?)\', \'js_only\'\n                        if not verify_ssl: continue\n                        break\n                    ssl_n = \'\' if verify_ssl else \' [SSL skipped]\'\n                    return text, f\'OK - {len(text):,} chars{ssl_n}\', None\n                except requests.exceptions.SSLError:\n                    last_error, last_reason = \'SSL error\', \'ssl_error\'\n                    if verify_ssl: continue\n                    break\n                except requests.exceptions.Timeout:\n                    last_error, last_reason = \'Timeout\', \'timeout\'; break\n                except requests.exceptions.TooManyRedirects:\n                    last_error, last_reason = \'Too many redirects\', \'connection_error\'; break\n                except requests.exceptions.ConnectionError as e:\n                    msg = str(e)\n                    reason = \'connection_error\'\n                    last_error = \'Domain not found\' if \'getaddrinfo\' in msg else msg[:60]\n                    last_reason = reason; break\n                except Exception as e:\n                    last_error, last_reason = str(e)[:60], \'unknown\'; break\n    return \'\', f\'FAILED: {last_error}\', last_reason\n\ndef run_analysis(url):\n    r = {}\n    norm     = normalize_url(url)\n    feats    = np.array(extract_url_features(norm), dtype=np.float32).reshape(1,-1)\n    seq      = char_tok.texts_to_sequences([norm])\n    padded   = pad_sequences(seq, maxlen=MAX_LEN, padding=\'post\', truncating=\'post\')\n    cnn_prob = float(cnn_model.predict(padded, verbose=0)[0][0])\n    h_input  = np.hstack((feats, [[cnn_prob]]))\n    url_prob = float(hybrid_model.predict_proba(h_input)[0][1])\n    r[\'url_prob\'] = url_prob\n    r[\'cnn_prob\'] = cnn_prob\n    raw_text, status, fail_reason = scrape_url(url)\n    r[\'scrape_status\'] = status\n    r[\'fail_reason\']   = fail_reason\n    if raw_text:\n        wc = len(raw_text.split())\n        if wc >= 500: q = \'high\'\n        elif wc >= 150: q = \'medium\'\n        elif wc >= 50: q = \'low\'\n        else: q = \'very_low\'\n        clean  = clean_html_text(raw_text)\n        seq2   = html_tok.texts_to_sequences([clean])\n        pad2   = pad_sequences(seq2, maxlen=HTML_MAX_LEN, padding=\'post\', truncating=\'post\')\n        hp     = float(html_model.predict(pad2, verbose=0)[0][0])\n        r[\'html_prob\']     = hp\n        r[\'scraped_chars\'] = len(raw_text)\n        r[\'text_quality\']  = q\n        r[\'penalty\']       = 0.0\n        uw, hw = {\'high\':(0.35,0.65),\'medium\':(0.35,0.65),\'low\':(0.55,0.45),\'very_low\':(0.65,0.35)}.get(q,(0.5,0.5))\n        final  = uw * url_prob + hw * hp\n        r[\'mode\'] = f\'URL({uw:.0%}) + HTML({hw:.0%}) [{q} quality]\'\n    else:\n        penalty = SCRAPE_FAIL_PENALTY.get(fail_reason, 0.0)\n        r[\'html_prob\']     = None\n        r[\'scraped_chars\'] = 0\n        r[\'text_quality\']  = \'scrape_failed\'\n        r[\'penalty\']       = penalty\n        final = min(1.0, url_prob + penalty)\n        reason_label = fail_reason or \'unknown\'\n        r[\'mode\'] = f\'URL only [{reason_label}] +{penalty:.2f} suspicion penalty\'\n    r[\'final_prob\'] = final\n    if   final >= 0.75: r[\'verdict\'], r[\'risk\'] = \'PHISHING\',   \'HIGH\'\n    elif final >= 0.50: r[\'verdict\'], r[\'risk\'] = \'SUSPICIOUS\',  \'MEDIUM\'\n    elif final >= 0.30: r[\'verdict\'], r[\'risk\'] = \'SUSPICIOUS\',  \'LOW\'\n    else:               r[\'verdict\'], r[\'risk\'] = \'LEGITIMATE\',  \'MINIMAL\'\n    return r\n\n# --- UI ---\nst.title(\'PhishGuard\')\nst.markdown(\'AI phishing detector: **CNN+XGBoost URL model** combined with **BiLSTM HTML content model**\')\nst.markdown(\'---\')\n\nwith st.expander(\'How does it work?\'):\n    st.markdown(\'\'\'\n**Step 1 — URL Analysis (always runs)**  \nThe URL is analysed by a CNN that reads it character-by-character, plus 15 structural\nfeatures (length, entropy, suspicious TLDs, hyphens, IP addresses, etc.).\nAn XGBoost model combines these into a URL phishing probability.\n\n**Step 2 — HTML Content Analysis (runs when scraping succeeds)**  \nThe page content is scraped with BeautifulSoup and analysed by a BiLSTM\nthat reads the visible text word-by-word. Phishing pages have distinctive\nlanguage patterns: urgency words, fake login prompts, credential requests.\n\n**Step 3 — Combining the results**  \nWhen scraping succeeds: `final = URL_weight * url_score + HTML_weight * html_score`  \nWeights depend on how much text was scraped (high quality = trust HTML more).  \n  \nWhen scraping fails: `final = url_score + suspicion_penalty`  \nThe penalty reflects how suspicious the failure type is:\n- SSL error: +0.15 (phishing sites often have bad certificates)\n- DNS failure / unreachable: +0.12 (may be down or fake)\n- Bot-blocked (403/429): +0.10 (legitimate sites rarely block all requests)\n- Timeout: +0.08 (slow / unresponsive)\n- JS-only page: +0.05 (common for legitimate SPAs too)\n    \'\'\')\n\nurl_input = st.text_input(\'Enter URL to analyse:\', placeholder=\'e.g. https://www.paypal.com\')\n\nif st.button(\'Analyse URL\', type=\'primary\') and url_input.strip():\n    with st.spinner(\'Analysing URL and scraping page content...\'):\n        r = run_analysis(url_input.strip())\n    verdict  = r[\'verdict\']\n    risk     = r[\'risk\']\n    pct      = r[\'final_prob\'] * 100\n    if verdict == \'PHISHING\':\n        st.error(f\'PHISHING DETECTED  |  {pct:.1f}% phishing probability  |  Risk: {risk}\')\n    elif risk == \'MEDIUM\':\n        st.warning(f\'SUSPICIOUS  |  {pct:.1f}% phishing probability  |  Risk: {risk}\')\n    elif risk == \'LOW\':\n        st.warning(f\'POSSIBLY SUSPICIOUS  |  {pct:.1f}% phishing probability  |  Risk: {risk}\')\n    else:\n        st.success(f\'LIKELY LEGITIMATE  |  {pct:.1f}% phishing probability  |  Risk: {risk}\')\n    st.markdown(\'### Score Breakdown\')\n    c1, c2, c3 = st.columns(3)\n    c1.metric(\'URL Model\', f"{r[\'url_prob\']*100:.1f}%", help=\'CNN + XGBoost URL pattern score\')\n    if r[\'html_prob\'] is not None:\n        c2.metric(\'HTML Model\', f"{r[\'html_prob\']*100:.1f}%", help=\'BiLSTM page content score\')\n    else:\n        pen = r[\'penalty\']\n        c2.metric(\'HTML Model\', \'N/A\', delta=f\'+{pen:.0%} penalty\' if pen > 0 else \'No penalty\',\n                  delta_color=\'inverse\', help=f\'Scrape failed: {r["fail_reason"]}\')\n    c3.metric(\'Final Score\', f\'{pct:.1f}%\', help=\'Combined weighted result\')\n    with st.expander(\'Full Technical Details\'):\n        st.write(f\'**Analysis mode:** {r["mode"]}\')\n        st.write(f\'**Scrape status:** {r["scrape_status"]}\')\n        st.write(f\'**Fail reason:** {r["fail_reason"]}\')\n        st.write(f\'**Characters scraped:** {r["scraped_chars"]}\')\n        st.write(f\'**Content quality:** {r["text_quality"]}\')\n        st.write(f\'**CNN raw prob:** {r["cnn_prob"]:.4f}\')\n        st.write(f\'**URL (XGBoost) prob:** {r["url_prob"]:.4f}\')\n        if r[\'html_prob\'] is not None:\n            st.write(f\'**HTML (BiLSTM) prob:** {r["html_prob"]:.4f}\')\n        if r[\'penalty\'] > 0:\n            st.write(f\'**Suspicion penalty added:** +{r["penalty"]:.2f} (reason: {r["fail_reason"]})\')\n        st.write(f\'**Final combined score:** {r["final_prob"]:.4f}\')\n    st.caption(\'Thresholds: >=75% Phishing | 50-74% Suspicious (medium) | 30-49% Suspicious (low) | <30% Legitimate\')\n'

with open('phishguard_app.py', 'w', encoding='utf-8') as f:
    f.write(gui_code)
print('phishguard_app.py written')
print('Run: streamlit run phishguard_app.py')
print()
print('STEP 24 COMPLETE')


phishguard_app.py written
Run: streamlit run phishguard_app.py

STEP 24 COMPLETE
